# 🌎 Mundo Autônomo — a corrida de nascimento (v0)

A Terra real (América do Sul) habitada por **8 empresas autônomas de IA**. 10 anos simulados, 120 ticks mensais, seed fixa. O notebook imprime a **crônica** e o **placar final como veredito** e salva `frames.json`/`chronicle.json`/`scoreboard.json` para o visualizador (`viewer/index.html` do repo).

Regra do fundador: simulação de verdade roda AQUI, no Kaggle — nunca na VM de desenvolvimento.

In [ ]:
# 1) Fonte do pacote: dataset privado 'pascalcoimbra/mundo-autonomo-src2'
#    (criado a partir do repo com scripts/kaggle_push_kernel.py --bundle-src)
import glob, os, shutil, subprocess, sys, zipfile

cands = glob.glob('/kaggle/input/**/mundo*.zip', recursive=True) + \
        glob.glob('/kaggle/input/*/src/mundo/__init__.py')
if os.path.isdir('/kaggle/input/mundo-autonomo-src2/src'):
    src_root = '/kaggle/input/mundo-autonomo-src2'
elif cands and cands[0].endswith('.zip'):
    zipfile.ZipFile(cands[0]).extractall('/kaggle/working/pkg')
    src_root = '/kaggle/working/pkg'
else:
    raise SystemExit('Fonte do pacote não encontrada: anexe o dataset '
                     'pascalcoimbra/mundo-autonomo-src2 a este kernel.')
print('pacote em', src_root)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', src_root], check=True)
import mundo
print('mundo', mundo.__version__, 'instalado')

## 2) A corrida de nascimento — 10 anos, 8 empresas, seed 42

In [ ]:
from mundo.engine import Engine

eng = Engine(seed=42, n_companies=8)
res = eng.run(120)   # 120 ticks mensais = 10 anos
eng.save_outputs('/kaggle/working')
print(f"ticks={res['ticks']}  frames={len(res['frames'])}  "
      f"eventos da crônica={len(res['chronicle'])}")

## 3) Veredito — a crônica (marcos) e o placar final

In [ ]:
marks = [e for e in res['chronicle']
         if e['kind'] in ('chain_link', 'bankruptcy', 'acquisition', 'record', 'god', 'end')]
print('=== MARCOS DA HISTÓRIA ===')
for e in marks:
    print(f"[{e['date']}] {e['text']}")
print()
print('=== VEREDITO: PLACAR FINAL ===')
for r in res['scoreboard']:
    print(f"{r['name']:<16} {r['archetype']:<16} "
          f"{'viva ' if r['alive'] else 'FALIDA'}  "
          f"US$ {r['cash']/1e6:>8.1f}M  elo {r['stage']}  "
          f"instalações {r['facilities']}")
alive = sum(1 for r in res['scoreboard'] if r['alive'])
top = max(res['scoreboard'], key=lambda r: r['stage'])
print(f"VEREDITO: {alive}/8 empresas vivas; maior elo alcançado: "
      f"{top['stage']} ({top['name']}); outputs em /kaggle/working "
      f"(frames.json p/ o viewer, chronicle.json, scoreboard.json).")